In [ ]:
import numpy as np
import pandas as pd
import os
import glob

def prepare_data():
    # Load park list
    with open(os.path.join('rules', 'parks.txt'), 'r') as f:
        parks = [line.strip() for line in f]

    df_list = []
    
    for park in parks:
        try:
            # Construct path and find files
            path = os.path.join('data', '*', f'wait{park.upper()}.csv')
            files = glob.glob(path)
            
            if not files:
                continue
            
            # Combine all CSVs for the current park
            df = pd.concat(pd.read_csv(f) for f in files)
            
            # Remove rows where all ride columns are NaN (assumes column 0 is timestamp)
            df.dropna(subset=df.columns[1:], how='all', inplace=True)
            
            # Convert first column to datetime object
            timestamps = pd.to_datetime(df.iloc[:, 0], format='mixed')
            
            # Extract features for Machine Learning
            df['day_of_week'] = timestamps.dt.dayofweek
            df['hour'] = timestamps.dt.hour
            
            # Cyclic encoding for time (helps model understand 23:00 is near 00:00)
            df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
            df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
            
            # Drop the original timestamp string column as ML models can't process it
            df.drop(df.columns[0], axis=1, inplace=True)
            df.drop(columns=['hour'], inplace=True)
            df = df.fillna(-1, inplace=True)

            # Save the processed data
            output_file = f'{park.upper()}test.csv'
            df.to_csv(output_file, index=False)

            df_list.append(df)
            return pd.concat(df_list)

        except Exception as e:
            print(f"Error processing {park}: {e}")
            continue

Error processing mk: 'NoneType' object has no attribute 'to_csv'
Error processing ep: 'NoneType' object has no attribute 'to_csv'
Error processing hs: 'NoneType' object has no attribute 'to_csv'
Error processing ak: 'NoneType' object has no attribute 'to_csv'
